# Fraud Detection Baselines



In [1]:
%pip install -q kagglehub xgboost scikit-learn joblib

In [2]:
import os
import time
import joblib
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

GLOBAL_SEED = 42
TARGET = "is_fraud"
TIME_COL = "trans_date_trans_time"
np.random.seed(GLOBAL_SEED)

## 1. Load the data and define chronological splits



In [3]:
data_dir = kagglehub.dataset_download("kartik2112/fraud-detection")

train_raw = pd.read_csv(os.path.join(data_dir, "fraudTrain.csv"), index_col=0)
test_raw = pd.read_csv(os.path.join(data_dir, "fraudTest.csv"), index_col=0)
all_raw = pd.concat([train_raw, test_raw], ignore_index=True)

required = {
    TARGET, TIME_COL, "amt", "category", "gender", "state", "city_pop",
    "lat", "long", "merch_lat", "merch_long", "dob", "job"
}
missing = required.difference(all_raw.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

all_raw[TIME_COL] = pd.to_datetime(all_raw[TIME_COL], errors="coerce")
all_raw = (
    all_raw.dropna(subset=[TIME_COL])
    .sort_values(TIME_COL)
    .reset_index(drop=True)
)

start_time = all_raw[TIME_COL].min()
cutoff_test = start_time + pd.DateOffset(months=10)
development_raw = all_raw.loc[all_raw[TIME_COL] < cutoff_test].copy()
test_raw_temporal = all_raw.loc[all_raw[TIME_COL] >= cutoff_test].copy()

val_cutoff = development_raw[TIME_COL].quantile(0.80)
train_raw_temporal = development_raw.loc[development_raw[TIME_COL] < val_cutoff].copy()
val_raw_temporal = development_raw.loc[development_raw[TIME_COL] >= val_cutoff].copy()

if train_raw_temporal.empty or val_raw_temporal.empty or test_raw_temporal.empty:
    raise ValueError("A chronological split is empty. Check the dataset date range.")

for name, frame in (
    ("Train", train_raw_temporal),
    ("Validation", val_raw_temporal),
    ("Final holdout", test_raw_temporal),
):
    print(
        f"{name}: rows={len(frame):,}, "
        f"{frame[TIME_COL].min()} -> {frame[TIME_COL].max()}, "
        f"fraud_rate={frame[TARGET].mean():.4%}"
    )

Using Colab cache for faster access to the 'fraud-detection' dataset.
Train: rows=570,695, 2019-01-01 00:00:18 -> 2019-08-31 00:51:49, fraud_rate=0.5844%
Validation: rows=142,674, 2019-08-31 00:51:55 -> 2019-10-31 23:59:34, fraud_rate=0.6343%
Final holdout: rows=1,139,025, 2019-11-01 00:00:59 -> 2020-12-31 23:59:34, fraud_rate=0.4751%


## 2. Feature engineering and leakage controls



In [4]:
def make_features(frame):
    d = frame.copy()
    d = d.drop(columns=["Unnamed: 0", "split"], errors="ignore")
    d[TIME_COL] = pd.to_datetime(d[TIME_COL], errors="coerce")
    d["hour"] = d[TIME_COL].dt.hour
    d["dayofweek"] = d[TIME_COL].dt.dayofweek
    d["month"] = d[TIME_COL].dt.month
    d["day"] = d[TIME_COL].dt.day
    d["is_weekend"] = (d["dayofweek"] >= 5).astype("int8")

    dob = pd.to_datetime(d["dob"], errors="coerce")
    d["age"] = ((d[TIME_COL] - dob).dt.days / 365.25).clip(18, 100)

    lat1, lon1 = np.radians(d["lat"]), np.radians(d["long"])
    lat2, lon2 = np.radians(d["merch_lat"]), np.radians(d["merch_long"])
    a = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    )
    d["distance_km"] = 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    drop_cols = [
        TARGET, TIME_COL, "trans_num", "cc_num", "first", "last", "street",
        "zip", "dob", "unix_time", "Unnamed: 0", "split"
    ]
    X = d.drop(columns=drop_cols, errors="ignore")
    y = pd.to_numeric(d[TARGET], errors="raise").astype("int8")
    for col in X.select_dtypes(include=["bool"]).columns:
        X[col] = X[col].astype("int8")
    return X, y, d[TIME_COL]

X_train_all, y_train_all, train_time = make_features(train_raw_temporal)
X_val, y_val, val_time = make_features(val_raw_temporal)
X_test, y_test, test_time = make_features(test_raw_temporal)

X_val = X_val.reindex(columns=X_train_all.columns)
X_test = X_test.reindex(columns=X_train_all.columns)

X_train_all = X_train_all.reset_index(drop=True)
y_train_all = y_train_all.reset_index(drop=True)
train_time = train_time.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
val_time = val_time.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
test_time = test_time.reset_index(drop=True)

print("Train features:", X_train_all.shape)
print("Validation features:", X_val.shape)
print("Final holdout features:", X_test.shape)
print("Fraud rates:")
print(f"Train:         {y_train_all.mean():.4%}")
print(f"Validation:    {y_val.mean():.4%}")
print(f"Final holdout: {y_test.mean():.4%}")

assert train_time.max() < val_time.min()
assert val_time.max() < test_time.min()

Train features: (570695, 19)
Validation features: (142674, 19)
Final holdout features: (1139025, 19)
Fraud rates:
Train:         0.5844%
Validation:    0.6343%
Final holdout: 0.4751%


## 3. Preprocessing and training sample



In [5]:
MAX_LEGIT_TRAIN = 250_000

numeric_cols = X_train_all.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train_all.columns.difference(numeric_cols).tolist()

fraud_idx = np.flatnonzero(y_train_all.to_numpy() == 1)
legit_idx = np.flatnonzero(y_train_all.to_numpy() == 0)
if MAX_LEGIT_TRAIN is not None and len(legit_idx) > MAX_LEGIT_TRAIN:
    legit_idx = np.random.RandomState(GLOBAL_SEED).choice(
        legit_idx, size=MAX_LEGIT_TRAIN, replace=False
    )
train_idx = np.concatenate([fraud_idx, legit_idx])
np.random.RandomState(GLOBAL_SEED).shuffle(train_idx)
X_fit = X_train_all.iloc[train_idx].copy()
y_fit = y_train_all.iloc[train_idx].copy()
X_eval, y_eval = X_test.copy(), y_test.copy()

print(f"Model training rows: {len(X_fit):,} | fraud={int(y_fit.sum()):,} | legit={int((y_fit == 0).sum()):,}")
print(f"Validation rows: {len(X_val):,} | fraud={int(y_val.sum()):,} | legit={int((y_val == 0).sum()):,}")

def make_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipe = Pipeline(numeric_steps)
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ], remainder="drop")

Model training rows: 253,335 | fraud=3,335 | legit=250,000
Validation rows: 142,674 | fraud=905 | legit=141,769


## 4. Metrics and threshold selection



In [6]:
def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


def best_f1_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        return 0.5
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.nanargmax(f1))])


def recall_at_fpr(y_true, scores, target_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, scores)
    eligible = tpr[fpr <= target_fpr]
    return float(eligible.max()) if len(eligible) else 0.0


def evaluate_scores(y_true, scores, threshold):
    pred = (np.asarray(scores) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "recall_at_1pct_fpr": recall_at_fpr(y_true, scores, 0.01),
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

## 5. Traditional model comparison

- XGBoost
- Random Forest
- Logisitic Regression

In [7]:
def xgb_pos_weight(y):
    y = np.asarray(y)
    return float((y == 0).sum() / max((y == 1).sum(), 1))


MODEL_GRIDS = {
    "logistic_regression": [
        {"C": 0.1},
        {"C": 1.0},
    ],
    "random_forest": [
        {"n_estimators": 200, "max_depth": 12, "min_samples_leaf": 2},
        {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 2},
    ],
    "xgboost": [
        {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.05},
        {"n_estimators": 400, "max_depth": 6, "learning_rate": 0.05},
    ],
}


def build_model(family, params, y):
    if family == "logistic_regression":
        estimator = LogisticRegression(
            C=params["C"],
            class_weight="balanced",
            max_iter=1000,
            solver="liblinear",
            random_state=GLOBAL_SEED,
        )
        preprocessor = make_preprocessor(scale_numeric=True)
    elif family == "random_forest":
        estimator = RandomForestClassifier(
            **params,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=GLOBAL_SEED,
        )
        preprocessor = make_preprocessor(scale_numeric=False)
    elif family == "xgboost":
        estimator = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=xgb_pos_weight(y),
            n_jobs=-1,
            random_state=GLOBAL_SEED,
        )
        preprocessor = make_preprocessor(scale_numeric=False)
    else:
        raise ValueError(f"Unknown model family: {family}")
    return Pipeline([("preprocess", preprocessor), ("model", estimator)])

In [ ]:
runs = []
fitted_candidates = {}
t_all = time.time()

for family, grid in MODEL_GRIDS.items():
    print(f"\n=== {family} ===")
    for run_i, params in enumerate(grid, start=1):
        start = time.time()
        model = build_model(family, params, y_fit)
        model.fit(X_fit, y_fit)
        val_scores = get_scores(model, X_val)
        threshold = best_f1_threshold(y_val.to_numpy(), val_scores)
        metrics = evaluate_scores(y_val.to_numpy(), val_scores, threshold)
        run_key = (family, run_i)
        fitted_candidates[run_key] = {
            "model": model,
            "params": params,
            "threshold": threshold,
            "val_scores": val_scores,
            "metrics": metrics,
        }
        runs.append({
            "family": family,
            "run": run_i,
            "params": params,
            **metrics,
            "fit_seconds": time.time() - start,
        })
        print(
            f"  {run_i}: AP={metrics['average_precision']:.5f}, "
            f"ROC-AUC={metrics['roc_auc']:.5f}, F1={metrics['f1']:.4f}, "
            f"time={time.time() - start:.1f}s"
        )

results = pd.DataFrame(runs).sort_values(
    "average_precision", ascending=False
).reset_index(drop=True)
print(f"\nSearch completed in {(time.time() - t_all) / 60:.1f} minutes.")
display(results[[
    "family", "run", "params", "average_precision", "roc_auc",
    "recall_at_1pct_fpr", "precision", "recall", "f1", "threshold", "fit_seconds",
]])


=== logistic_regression ===
  1: AP=0.06945, ROC-AUC=0.83542, F1=0.1962, time=9.4s
  2: AP=0.01974, ROC-AUC=0.74795, F1=0.0692, time=14.2s

=== random_forest ===
  1: AP=0.60237, ROC-AUC=0.94608, F1=0.5880, time=51.4s


## 6. Compare validation results


In [ ]:
best_by_family = results.loc[
    results.groupby("family")["average_precision"].idxmax()
]
display(best_by_family[[
    "family", "average_precision", "roc_auc", "recall_at_1pct_fpr",
    "precision", "recall", "f1", "threshold",
]].sort_values("average_precision", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
best_by_family.plot(
    x="family", y="average_precision", kind="bar", ax=axes[0], legend=False
)
axes[0].set_title("Validation Average Precision")
axes[0].set_ylabel("Average precision")
axes[0].tick_params(axis="x", rotation=25)
best_by_family.plot(
    x="family", y="recall_at_1pct_fpr", kind="bar", ax=axes[1], legend=False
)
axes[1].set_title("Validation Recall at 1% FPR")
axes[1].set_ylabel("Recall")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 7. Final holdout evaluation



In [ ]:
winner_row = results.iloc[0]
winner_key = (winner_row["family"], int(winner_row["run"]))
winner = fitted_candidates[winner_key]
final_model = winner["model"]
winner_family = winner_key[0]
winner_params = winner["params"]
threshold = float(winner["threshold"])

print("Selected model:", winner_family)
print("Parameters:", winner_params)
print("Validation AP:", round(float(winner_row["average_precision"]), 6))
print("Threshold from validation:", threshold)

eval_scores = get_scores(final_model, X_eval)
final_metrics = evaluate_scores(y_eval.to_numpy(), eval_scores, threshold)
print("\nFinal holdout metrics:")
for metric_name, metric_value in final_metrics.items():
    print(f"{metric_name:>24}: {metric_value}")

eval_pred = (eval_scores >= threshold).astype(int)
print("\nClassification report (final holdout):")
print(classification_report(y_eval, eval_pred, digits=4, zero_division=0))

## 8. Final holdout plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_eval,
    eval_pred,
    labels=[0, 1],
    display_labels=["Legitimate", "Fraud"],
    values_format=",",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Final holdout confusion matrix")
PrecisionRecallDisplay.from_predictions(
    y_eval, eval_scores, ax=axes[1], name=winner_family
)
axes[1].set_title("Final holdout precision-recall curve")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_eval, eval_scores, name=winner_family)
plt.title("Final holdout ROC curve")
plt.show()

## 9. Save the fitted pipeline and evaluation metadata



In [ ]:
MODEL_PATH = Path("kaggle_fraud_baseline.joblib")
bundle = {
    "model": final_model,
    "model_family": winner_family,
    "params": winner_params,
    "threshold": threshold,
    "feature_columns": X_train_all.columns.tolist(),
    "validation_metrics": winner["metrics"],
    "final_test_metrics": final_metrics,
    "target": TARGET,
    "dataset": "Kartik2112 Kaggle Fraud Detection",
}
joblib.dump(bundle, MODEL_PATH, compress=3)
print(f"Saved {MODEL_PATH.resolve()} ({MODEL_PATH.stat().st_size / 1e6:.2f} MB)")

reloaded = joblib.load(MODEL_PATH)
scores_check = get_scores(reloaded["model"], X_eval)
assert np.allclose(scores_check, eval_scores), "Reloaded model scores differ."
print("Reload check passed.")